# Decision Tree vs Random Forest — Obesity Levels (UCI)

Compares a tuned **Decision Tree** and a tuned **Random Forest** on the UCI dataset
*Estimation of obesity levels based on eating habits and physical condition*.

Two scenarios are evaluated:

- **A) With Height and Weight.** The class label is derived from BMI = Weight / Height², so this setting leaks the target.
- **B) Without Height and Weight.** The realistic setting: predict from eating habits and physical condition only.

**Setup:** put `ObesityDataSet_raw_and_data_sinthetic.csv` in a `data/` folder next to this notebook.
Requires: `pandas`, `scikit-learn`, `matplotlib`, `seaborn`.

## 1. Imports and configuration

In [ ]:
import os
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score)
from sklearn.model_selection import (GridSearchCV, StratifiedKFold,
                                     cross_val_score, train_test_split)
from sklearn.tree import DecisionTreeClassifier, plot_tree

warnings.filterwarnings("ignore")
%matplotlib inline

RANDOM_STATE = 42
DATA_PATH = "data/ObesityDataSet_raw_and_data_sinthetic.csv"
OUT = "results"
os.makedirs(OUT, exist_ok=True)

# Logical class order (from underweight to obesity type III)
CLASS_ORDER = ["Insufficient_Weight", "Normal_Weight", "Overweight_Level_I",
               "Overweight_Level_II", "Obesity_Type_I", "Obesity_Type_II",
               "Obesity_Type_III"]

## 2. Load and preprocess

- Drop duplicate rows
- Binary columns (yes/no, Gender) → 0/1
- `CAEC` and `CALC` → ordinal (`no` < `Sometimes` < `Frequently` < `Always`)
- `MTRANS` → one-hot
- No scaling needed: both models are tree-based

In [ ]:
def load_and_preprocess(path):
    df = pd.read_csv(path)
    n_before = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Duplicate rows removed: {n_before - len(df)}  |  final shape: {df.shape}")

    for col in ["family_history_with_overweight", "FAVC", "SMOKE", "SCC"]:
        df[col] = (df[col] == "yes").astype(int)
    df["Gender"] = (df["Gender"] == "Male").astype(int)

    ordinal_map = {"no": 0, "Sometimes": 1, "Frequently": 2, "Always": 3}
    for col in ["CAEC", "CALC"]:
        df[col] = df[col].map(ordinal_map)

    df = pd.get_dummies(df, columns=["MTRANS"], prefix="MTRANS", dtype=int)

    y = df["NObeyesdad"]
    X = df.drop(columns="NObeyesdad")
    return X, y


X, y = load_and_preprocess(DATA_PATH)
y.value_counts()

## 3. Training, tuning and evaluation helpers

In [ ]:
def tune_and_evaluate(X, y, scenario_name):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    models = {
        "Decision Tree": (
            DecisionTreeClassifier(random_state=RANDOM_STATE),
            {"max_depth": [3, 5, 7, 10, 15, None],
             "min_samples_split": [2, 5, 10, 20],
             "min_samples_leaf": [1, 2, 5, 10],
             "criterion": ["gini", "entropy"]},
        ),
        "Random Forest": (
            RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
            {"n_estimators": [100, 300],
             "max_depth": [10, 20, None],
             "min_samples_leaf": [1, 2, 5],
             "max_features": ["sqrt", "log2"]},
        ),
    }

    results, fitted, preds = [], {}, {}
    for name, (est, grid) in models.items():
        print(f"\n[{scenario_name}] Tuning {name} ...")
        gs = GridSearchCV(est, grid, cv=cv, scoring="f1_macro", n_jobs=-1)
        t0 = time.time()
        gs.fit(X_train, y_train)
        train_time = time.time() - t0
        best = gs.best_estimator_

        t0 = time.time()
        y_pred = best.predict(X_test)
        pred_time = time.time() - t0

        # CV on the full data with the best hyperparameters (stability check)
        cv_scores = cross_val_score(best, X, y, cv=cv, scoring="accuracy", n_jobs=-1)

        train_acc = accuracy_score(y_train, best.predict(X_train))
        test_acc = accuracy_score(y_test, y_pred)
        results.append({
            "Scenario": scenario_name,
            "Model": name,
            "Best Params": gs.best_params_,
            "Train Acc": round(train_acc, 4),
            "Test Acc": round(test_acc, 4),
            "Test F1 (macro)": round(f1_score(y_test, y_pred, average="macro"), 4),
            "Test F1 (weighted)": round(f1_score(y_test, y_pred, average="weighted"), 4),
            "CV Acc (mean)": round(cv_scores.mean(), 4),
            "CV Acc (std)": round(cv_scores.std(), 4),
            "Overfit Gap (train-test)": round(train_acc - test_acc, 4),
            "Tuning Time (s)": round(train_time, 1),
            "Predict Time (ms)": round(pred_time * 1000, 1),
        })
        fitted[name] = best
        preds[name] = y_pred
        print(f"  Best params: {gs.best_params_}")
        print(f"  Test Acc={test_acc:.4f}  CV={cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
        print(classification_report(y_test, y_pred, labels=CLASS_ORDER, digits=3))

    return results, fitted, preds, X_train, X_test, y_train, y_test

## 4. Plotting helpers

In [ ]:
def plot_confusion_matrices(preds, y_test, tag):
    fig, axes = plt.subplots(1, 2, figsize=(17, 7))
    for ax, (name, y_pred) in zip(axes, preds.items()):
        cm = confusion_matrix(y_test, y_pred, labels=CLASS_ORDER)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False,
                    xticklabels=CLASS_ORDER, yticklabels=CLASS_ORDER)
        ax.set_title(f"{name}  (acc={accuracy_score(y_test, y_pred):.3f})")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")
        ax.tick_params(axis="x", rotation=45)
    plt.suptitle(f"Confusion Matrix - {tag}", fontsize=14)
    plt.tight_layout()
    plt.savefig(f"{OUT}/confusion_{tag}.png", dpi=130)
    plt.show()


def plot_feature_importance(fitted, feature_names, tag, top=12):
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    for ax, (name, model) in zip(axes, fitted.items()):
        imp = pd.Series(model.feature_importances_, index=feature_names)
        imp = imp.sort_values(ascending=True).tail(top)
        imp.plot.barh(ax=ax, color="#3b7dd8" if "Tree" in name else "#2e9e5b")
        ax.set_title(name)
        ax.set_xlabel("Importance")
    plt.suptitle(f"Feature Importance - {tag}", fontsize=14)
    plt.tight_layout()
    plt.savefig(f"{OUT}/feature_importance_{tag}.png", dpi=130)
    plt.show()


def plot_tree_figure(tree_model, feature_names, tag, max_depth=3):
    plt.figure(figsize=(22, 10))
    plot_tree(tree_model, feature_names=feature_names,
              class_names=list(tree_model.classes_), filled=True, rounded=True,
              max_depth=max_depth, fontsize=8)
    plt.title(f"Decision Tree (top {max_depth} levels) - {tag}")
    plt.tight_layout()
    plt.savefig(f"{OUT}/decision_tree_{tag}.png", dpi=110)
    plt.show()


def plot_depth_curve(X, y, tag):
    """Overfitting behaviour: train/test accuracy vs. tree depth."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
    rows = []
    for d in range(1, 21):
        dt = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE).fit(X_train, y_train)
        rf = RandomForestClassifier(max_depth=d, n_estimators=200, random_state=RANDOM_STATE,
                                    n_jobs=-1).fit(X_train, y_train)
        rows.append([d, dt.score(X_train, y_train), dt.score(X_test, y_test),
                     rf.score(X_train, y_train), rf.score(X_test, y_test)])
    r = pd.DataFrame(rows, columns=["depth", "DT train", "DT test", "RF train", "RF test"])
    plt.figure(figsize=(9, 5))
    plt.plot(r.depth, r["DT train"], "--", color="#3b7dd8", label="DT train")
    plt.plot(r.depth, r["DT test"], "-", color="#3b7dd8", label="DT test")
    plt.plot(r.depth, r["RF train"], "--", color="#2e9e5b", label="RF train")
    plt.plot(r.depth, r["RF test"], "-", color="#2e9e5b", label="RF test")
    plt.xlabel("max_depth")
    plt.ylabel("Accuracy")
    plt.title(f"Accuracy vs Depth - {tag}")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{OUT}/depth_curve_{tag}.png", dpi=130)
    plt.show()

## 5. Scenario A — with Height and Weight

> Note: the label is a function of BMI, so high scores here mostly reflect target leakage.

In [ ]:
tag_A = "A_with_Height_Weight"
X_A = X
res_A, fitted_A, preds_A, _, _, _, y_test_A = tune_and_evaluate(X_A, y, tag_A)

In [ ]:
plot_confusion_matrices(preds_A, y_test_A, tag_A)
plot_feature_importance(fitted_A, X_A.columns, tag_A)

In [ ]:
plot_tree_figure(fitted_A["Decision Tree"], list(X_A.columns), tag_A)
plot_depth_curve(X_A, y, tag_A)

## 6. Scenario B — without Height and Weight

The realistic setting: only eating habits and physical condition are used.

In [ ]:
tag_B = "B_without_Height_Weight"
X_B = X.drop(columns=["Height", "Weight"])
res_B, fitted_B, preds_B, _, _, _, y_test_B = tune_and_evaluate(X_B, y, tag_B)

In [ ]:
plot_confusion_matrices(preds_B, y_test_B, tag_B)
plot_feature_importance(fitted_B, X_B.columns, tag_B)

In [ ]:
plot_tree_figure(fitted_B["Decision Tree"], list(X_B.columns), tag_B)
plot_depth_curve(X_B, y, tag_B)

## 7. Final comparison

In [ ]:
results_df = pd.DataFrame(res_A + res_B)
results_df.to_csv(f"{OUT}/comparison_results.csv", index=False)
results_df.drop(columns="Best Params")

In [ ]:
metrics = ["Test Acc", "Test F1 (macro)", "CV Acc (mean)"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, (scen, g) in zip(axes, results_df.groupby("Scenario", sort=False)):
    g = g.set_index("Model")[metrics]
    g.T.plot.bar(ax=ax, color=["#3b7dd8", "#2e9e5b"], rot=0)
    ax.set_title(scen, fontsize=10)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    for c in ax.containers:
        ax.bar_label(c, fmt="%.3f", fontsize=8)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=2, fontsize=9, frameon=False)
plt.suptitle("Decision Tree vs Random Forest", fontsize=14)
plt.tight_layout()
plt.savefig(f"{OUT}/summary_comparison.png", dpi=130)
plt.show()

## 8. Interpretation

Re-run the notebook and check the table above; in the run documented in the README the pattern was:

1. **Scenario A:** both models score above 95%, and the gap (~1 point) is within the noise of the split/CV, so neither can be called better. The high accuracy is mostly due to target leakage.
2. **Scenario B:** Random Forest is about 10 points better than the single tree (≈85% vs ≈75%), and the gap holds under cross-validation. Averaging many trees reduces variance; a single tree overfits much more on this noisier task.
3. **Hardest classes** in B are `Normal_Weight` and `Overweight_Level_I/II`, whose boundaries are close. `Obesity_Type_III` is almost perfectly recognised.
4. **Cost:** Random Forest tunes and predicts more slowly, but a single tree is easy to plot and interpret.

**Limitations:** about 77% of the dataset is synthetic (SMOTE), so near-duplicate samples can land in both train and test and make scores optimistic. Only one train/test split and seed were used (CV partly compensates), and the hyperparameter grids are small.